# 07.01 · Text embeddings

**Chapter:** Embeddings & Vector Search · **Milestone:** M3

## Goals
- [ ] What an embedding captures (meaning, topic, paraphrase) and what it doesn't (negation, exact numbers, rare names)
- [ ] Run open embedding models locally (sentence-transformers) and API models (Gemini, OpenAI…)
- [ ] Normalisation, dimensions, Matryoshka truncation, query vs document prefixes
- [ ] Choose a model with MTEB as a filter and *your* data as the judge

## Resources
- [Sentence Transformers docs](https://sbert.net/) — free
- [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) — free
- [Vicki Boykis – What are embeddings? (free book)](https://vickiboykis.com/what_are_embeddings/) — free

| Option | Examples | Notes |
|---|---|---|
| **Open, local** | `BAAI/bge-*`, `intfloat/e5-*`, `nomic-embed-text`, `Qwen3-Embedding`, `all-MiniLM-L6-v2`, `embeddinggemma` | free, private, fast on CPU/MPS for small models |
| **API** | OpenAI `text-embedding-3-*`, Gemini `gemini-embedding-001`, Voyage, Cohere, Jina, Mistral | strong multilingual quality, no infra, per-token cost |
| **Via local servers** | Ollama `/api/embed`, LM Studio, vLLM, TEI (Hugging Face Text Embeddings Inference) | serve open models behind an API |

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer

minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # 22M params, 384 dims
bge = SentenceTransformer("BAAI/bge-small-en-v1.5")  # 33M params, 384 dims
print(minilm.get_sentence_embedding_dimension(), bge.get_sentence_embedding_dimension())

## 1. Sentences in, vectors out
Normalise to unit length; then cosine similarity is a dot product (01.02).

In [ ]:
sentences = [
    "How do I reset my password?",
    "I forgot my login credentials.",
    "What's the weather like in Lima?",
    "Is it going to rain in Peru's capital?",
    "The cat sat on the mat.",
]
E = bge.encode(sentences, normalize_embeddings=True)
S = E @ E.T
plt.imshow(S, cmap="viridis", vmin=0, vmax=1); plt.colorbar()
plt.xticks(range(5), [s[:18] for s in sentences], rotation=45, ha="right"); plt.yticks(range(5), [s[:18] for s in sentences])
plt.title("cosine similarity (bge-small)"); plt.show()

Paraphrases with **no words in common** score high: "reset my password" ≈ "forgot my login credentials". That's what keyword search misses.

## 2. What embeddings get wrong
They compress meaning into one vector: topic dominates, details blur. Classic failure modes:

In [ ]:
pairs = [
    ("I love this phone", "I do not love this phone"),  # negation
    ("Flight to Lima on May 3rd", "Flight to Lima on May 30th"),  # numbers
    ("Error code E1043 when saving", "Error code E7781 when saving"),  # identifiers
    ("Apple released a new phone", "I ate an apple for lunch"),  # ambiguity (should be LOW)
]
for a, b in pairs:
    ea, eb = bge.encode([a, b], normalize_embeddings=True)
    print(f"{ea @ eb:.2f}  {a!r} vs {b!r}")

Very high similarity for opposite meanings and different IDs → embeddings alone can't do exact matching. Hybrid search with BM25 (07.04)
and rerankers (07.05) fix much of this.

## 3. Query vs document: prefixes / instructions
Many retrieval models are trained with asymmetric prefixes: short queries vs long passages.
bge: `"Represent this sentence for searching relevant passages: "` on queries. e5: `"query: "` / `"passage: "` (required). Check each model card.
Absolute scores shift; what matters is whether the **ranking** on your eval set improves — measure it (exercise in 07.04).

In [ ]:
query = "symptoms of altitude sickness"
docs = ["Headache, nausea and dizziness are common above 2,500 metres.", "Altitude is measured in metres above sea level."]
for prefix in ["", "Represent this sentence for searching relevant passages: "]:
    q = bge.encode([prefix + query], normalize_embeddings=True)[0]
    print(f"prefix={bool(prefix)!s:5}", [round(float(q @ d), 3) for d in bge.encode(docs, normalize_embeddings=True)])

## 4. Embedding space has structure
Embed sentences from 4 topics and project with UMAP (02.05). Good models separate topics without being told about them.

In [ ]:
import umap

topics = {
    "cooking": ["How long do I boil an egg?", "Best way to caramelise onions", "Recipe for ceviche", "How to knead bread dough",
                "Why is my sauce too salty?", "Substitute for eggs in baking", "How to sharpen a kitchen knife", "Cooking quinoa perfectly"],
    "programming": ["How to reverse a list in Python", "What is a race condition?", "Fix a segmentation fault in C", "Explain recursion simply",
                    "Difference between git merge and rebase", "How do Python decorators work", "What is a REST API?", "Debugging a memory leak"],
    "sports": ["Who won the last World Cup?", "How to improve my 10k running time", "Rules of the offside in football", "Best stretches after a run",
               "How many sets in a tennis match?", "Training plan for a marathon", "What is a hat-trick?", "How to swim butterfly stroke"],
    "finance": ["How does compound interest work?", "What is an index fund?", "Should I pay off debt or invest?", "How are taxes on dividends calculated?",
                "What is inflation?", "How to build an emergency fund", "Difference between stocks and bonds", "What is a credit score?"],
}
texts = [t for ts in topics.values() for t in ts]
labels = [name for name, ts in topics.items() for _ in ts]
Z = umap.UMAP(n_neighbors=8, random_state=0).fit_transform(bge.encode(texts, normalize_embeddings=True))
for name in topics:
    idx = [i for i, l in enumerate(labels) if l == name]
    plt.scatter(Z[idx, 0], Z[idx, 1], label=name)
plt.legend(); plt.title("32 questions, 4 topics, embedded with bge-small + UMAP"); plt.show()

## 5. Dimensions and Matryoshka truncation
Bigger vectors = more storage and slower search. **Matryoshka**-trained models (OpenAI `text-embedding-3-*`, Gemini, nomic, many new open
models) keep most quality when you truncate to the first N dims (then re-normalise). Storage math for 10M chunks in float32:

In [ ]:
for dims in [384, 768, 1536, 3072]:
    print(f"{dims:5} dims x 10M vectors = {dims * 4 * 10_000_000 / 1e9:6.1f} GB (float32) | {dims * 10_000_000 / 1e9:5.1f} GB (int8) | {dims * 10_000_000 / 8 / 1e9:4.1f} GB (binary)")

## 6. API embeddings (Gemini — free tier)
Most providers expose `/embeddings` in the OpenAI format too.

In [ ]:
from aieng.llm import client

if os.getenv("GEMINI_API_KEY"):
    gc, _ = client("gemini")
    r = gc.embeddings.create(model="gemini-embedding-001", input=sentences[:2], dimensions=768)
    a, b = (np.array(d.embedding) for d in r.data)
    a, b = a / np.linalg.norm(a), b / np.linalg.norm(b)
    print(f"gemini-embedding-001: {len(r.data[0].embedding)} dims | sim(password, credentials) = {a @ b:.3f}")
else:
    print("GEMINI_API_KEY not set — skipping (free key at aistudio.google.com)")

Ollama alternative: `ollama pull nomic-embed-text`, then `client("ollama")[0].embeddings.create(model="nomic-embed-text", input=[...])`.

## 7. Throughput
Embedding the corpus is a batch job; queries are single, latency-sensitive calls.

In [ ]:
import time

batch = texts * 32  # ~1000 sentences
for name, m in [("all-MiniLM-L6-v2", minilm), ("bge-small-en-v1.5", bge)]:
    start = time.perf_counter(); m.encode(batch, batch_size=64); secs = time.perf_counter() - start
    print(f"{name:20} {len(batch) / secs:6.0f} sentences/s on {m.device}")

## Exercises
1. Embed the same 32 questions with MiniLM, bge-small and (if you have a key) Gemini. Which separates topics best? Measure it: k-means (k=4)
   + adjusted Rand index against the labels (02.05).
2. Matryoshka test: with Gemini (`dimensions=` 128/256/768/3072) or `nomic-embed-text`, measure how the top-1 neighbour changes as you shrink dims.
3. Write 10 pairs in Spanish (or your language) + English translations. Does bge-small (English-only) match them? Try
   `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` or a multilingual API model.

In [ ]:
# your code here